<a href="https://colab.research.google.com/github/steryams/Laboratory_Exam/blob/main/Yamomo_Laboratory_Exam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

# Load the telecom churn dataset
df = pd.read_csv('telecom_churn (1).csv')

# Inspect basic information
print(f"Dataset Shape: {df.shape}")
df.head()

Dataset Shape: (3333, 21)


,state,account length,area code,phone number,international plan,voice mail plan,number vmail messages,total day minutes,total day calls,total day charge,...,total eve calls,total eve charge,total night minutes,total night calls,total night charge,total intl minutes,total intl calls,total intl charge,customer service calls,churn
0,KS,128,415,382-4657,no,yes,25,265.1,110,45.07,...,99,16.78,244.7,91,11.01,10.0,3,2.70,1,False
1,OH,107,415,371-7191,no,yes,26,161.6,123,27.47,...,103,16.62,254.4,103,11.45,13.7,3,3.70,1,False
2,NJ,137,415,358-1921,no,no,0,243.4,114,41.38,...,110,10.30,162.6,104,7.32,12.2,5,3.29,0,False
3,OH,84,408,375-9999,yes,no,0,299.4,71,50.90,...,88,5.26,196.9,89,8.86,6.6,7,1.78,2,False
4,OK,75,415,330-6626,yes,no,0,166.7,113,28.34,...,122,12.61,186.9,121,8.41,10.1,3,2.73,3,False


In [2]:
# 1. Check for missing values across all columns
missing_counts = df.isnull().sum()
print("Missing values per column:\n", missing_counts[missing_counts > 0])

# 2. Check for duplicate rows
print("Duplicate rows count:", df.duplicated().sum())

# 3. Drop uninformative features
# 'phone number' is a unique identifier and adds no predictive value
X = df.drop(columns=['phone number'])
y = df.drop(columns=['phone number'])['churn'].astype(int)

Missing values per column:
 Series([], dtype: int64)
Duplicate rows count: 0


In [3]:
# Separate features and target
X = df.drop(columns=['phone number', 'churn'])
y = df['churn'].astype(int)

# Identify categorical and numerical columns
cat_cols = ['state', 'international plan', 'voice mail plan', 'area code']
num_cols = [c for c in X.columns if c not in cat_cols]

# One-hot encode categorical variables in pandas
X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True)
print(f"Features shape after One-Hot Encoding: {X_encoded.shape}")

Features shape after One-Hot Encoding: (3333, 69)


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# 1. Define feature subsets
cat_cols = ['state', 'international plan', 'voice mail plan', 'area code']
num_cols = [c for c in X.columns if c not in cat_cols]

# 2. Split data into train and test sets (stratified to preserve class ratio)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Build preprocessing transformers for numeric and categorical pipelines
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), # Handles any future missing numeric values
    ('scaler', StandardScaler())                    # Normalizes numerical features
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')), # Handles missing categorical values
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))      # One-hot encoding
])

# 4. Combine into a single ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols)
    ]
)

# 5. Fit the preprocessor on the training data and transform both sets
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed Training Feature Matrix Shape:", X_train_processed.shape)

Processed Training Feature Matrix Shape: (2666, 69)


In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# 1. Load the telecom dataset
df = pd.read_csv('telecom_churn (1).csv')

# 2. Separate features (dropping unique ID phone number) and target variable
X = df.drop(columns=['phone number', 'churn'])
y = df['churn'].astype(int)

# 3. Partition data into stratified train and test sets (80% / 20% split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Map and separate categorical and numerical feature names for preprocessing
cat_cols = ['state', 'international plan', 'voice mail plan', 'area code']
num_cols = [c for c in X.columns if c not in cat_cols]

# 5. Build the pipeline ColumnTransformer for standardization and one-hot encoding
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ]
)

# 6. Define the three models to compare
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42)
}

# 7. Train, predict target classes (True/False) & probabilities, and calculate metrics
evaluation_results = {}
predictions_df = pd.DataFrame({'Actual Churn': y_test.astype(bool)})

for name, clf in models.items():
    # Combine preprocessing and model into an end-to-end Pipeline
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', clf)
    ])

    # Fit on training data
    pipeline.fit(X_train, y_train)

    # Predict binary targets (True/False class predictions) and soft probabilities
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    # Save predictions to dataframe for inspection
    predictions_df[f'{name} Predicted Class'] = y_pred.astype(bool)
    predictions_df[f'{name} Probability'] = y_prob

    # Calculate key evaluation metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    evaluation_results[name] = {
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1 Score': f1,
        'ROC-AUC': auc
    }

# 8. Render a sample of the predicted targets and probabilities
print("=== Sample Predictions (Target Classes as True/False & Probabilities) ===")
display(predictions_df.head(10))

# 9. Format and print the comparative summary table
summary_df = pd.DataFrame(evaluation_results).T
print("\n=== Comparative Model Performance Summary Table ===")
display(summary_df.round(4))

=== Sample Predictions (Target Classes as True/False & Probabilities) ===


,Actual Churn,Logistic Regression Predicted Class,Logistic Regression Probability,Decision Tree Predicted Class,Decision Tree Probability,Random Forest Predicted Class,Random Forest Probability
601,True,False,0.180751,True,1.0,True,0.78
2050,False,False,0.124919,False,0.0,False,0.02
3200,False,False,0.111847,False,0.0,False,0.30
1953,False,False,0.131832,False,0.0,False,0.04
1119,False,False,0.359478,False,0.0,False,0.07
2204,False,False,0.055031,False,0.0,False,0.03
1888,False,False,0.280527,False,0.0,False,0.06
738,False,False,0.114126,False,0.0,False,0.05
3087,False,False,0.033666,False,0.0,False,0.06
539,False,False,0.102921,False,0.0,False,0.05



=== Comparative Model Performance Summary Table ===


,Accuracy,Precision,Recall,F1 Score,ROC-AUC
Logistic Regression,0.8591,0.5333,0.2474,0.3380,0.8086
Decision Tree,0.9025,0.6702,0.6495,0.6597,0.7975
Random Forest,0.9355,0.9091,0.6186,0.7362,0.8911
